# NLLB 600M: Plain BPE vs weighted penalty32 Morph-BPE
Evaluate the current saved 30-epoch models, without training or changing weights. Both conditions use identical rows and decoding settings, sequentially on one GPU. BLEU and chrF++ score decoded Filipino text against the CSV references.

**Input:** `kaggle_bleu_chrf_inputs_30epochs_weighted32.zip`, containing current Plain BPE, weighted penalty32 Morph-BPE and their matching CSV. Attach only this package once; Kaggle may expand it or keep it zipped. No separate training-input ZIP or CSV upload is needed.

Enable GPU and Internet. Run section 1 once, restart the session, then continue from section 2. The first load downloads the pinned frozen NLLB base. Both custom source tables and tokenizers are already trained. The Morph tokenizer is byte-identical to the Tokenization tab's penalty32 artifact.

Use SPLIT='test', MAX_EXAMPLES=None and the new OUTPUT_NAME for full evaluation. Do not attach the previous hard-constrained `paired_evaluation_v2_backup.zip` as resume input. This notebook rejects old checkpoints and changed resume identities. Resume only a backup made by this revised notebook with these inputs.

Output filenames, metric signatures and variable-table columns match the previous evaluation. Numerical scores must be measured for the current weighted model and are not expected to equal the old Morph scores. Test results are for final reporting, not tuning.


## 1. Install compatible dependencies, then restart the session
Keep Kaggle's installed PyTorch/CUDA. This dedicated text-evaluation notebook removes optional vision/UI libraries that conflict with this Transformers environment. Do not run this cell again after restarting.

In [ ]:
%pip -q uninstall -y diffusers gradio timm torchvision torchaudio
%pip -q install "transformers==4.48.3" "tokenizers==0.21.0" "sentencepiece==0.2.0" "safetensors==0.5.3" "huggingface-hub==0.28.1" "sacrebleu==2.5.1" tqdm
# Restart the Kaggle session now, then start at section 2.


## 2. Settings and GPU check
Leave MAX_EXAMPLES=None for the full test evaluation. For a short pipeline check use MAX_EXAMPLES=5 and OUTPUT_NAME="bleu_chrf_smoke"; those scores are not final thesis results. Restore None and use a fresh output name for the full run.

In [ ]:
from pathlib import Path
import torch
SPLIT = 'test'
MAX_EXAMPLES = None
OUTPUT_NAME = 'bleu_chrf_weighted32_full_test'
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working') / OUTPUT_NAME
WORK.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), 'Enable a GPU accelerator in Kaggle settings.'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))


## 3. Write the self-contained evaluation utility
No repository clone is required. This helper verifies bundle identity, evaluates the matching source embeddings and exports reproducible corpus metrics.

In [ ]:
%%writefile /kaggle/working/paired_evaluation_v2.py
"""Embedded in the Kaggle notebook; evaluation only, no optimizer or training."""
from pathlib import Path
import csv, gc, hashlib, importlib.util, json, os, shutil, sys, time, zipfile
import torch
import transformers
import sacrebleu
from sacrebleu.metrics import BLEU, CHRF
from safetensors.torch import load_file
from tqdm.auto import tqdm

CONDITIONS = ('plain_bpe', 'morph_bpe')
EXPECTED_CHECKPOINTS = {'plain_bpe': {'weights': '7de78e7e02f01b481b3e94328218782610b283e6a701aa09246c2836601bda02', 'tokenizer': '4098512366353a8d0d0793b11a62eb53f92ea442972b29bb042a4cef2069fa52'}, 'morph_bpe': {'weights': '6d7721111fa35f7b11e97b5e832e32fa81f28f58d23fdfdfd60fbdcdf60f67fc', 'tokenizer': '3021219634a9f510489405526fca3e8d60823089f95673bcf9c0f9a4e4c7e972'}}

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()

def save_json(path, data):
    Path(path).write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')

def extract(archive, destination):
    destination = Path(destination); destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            if not (destination / item.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Unsafe archive path')
            if (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Archive symlinks are not supported')
        z.extractall(destination)

def discover(input_root, work):
    input_root, work = Path(input_root), Path(work)
    bundles = {}; data_files = list(input_root.rglob('kpm_tgl_cleaned.csv'))
    def register(path):
        info = json.loads((path / 'manifest.json').read_text(encoding='utf-8'))
        condition = info.get('identity', {}).get('condition')
        if condition not in CONDITIONS: return
        if condition in bundles: raise ValueError(f'Duplicate {condition}: attach only one copy of each model.')
        bundles[condition] = path
    for p in input_root.rglob('manifest.json'): register(p.parent)
    for index, archive in enumerate(sorted(input_root.rglob('*.zip'))):
        with zipfile.ZipFile(archive) as z:
            candidates = [name for name in z.namelist() if name.endswith('manifest.json') and Path(name).name == 'manifest.json']
            valid = [name for name in candidates if json.loads(z.read(name)).get('identity', {}).get('condition') in CONDITIONS]
        if not valid: continue
        dest = work.parent / 'extracted_inputs' / str(index)
        extract(archive, dest)
        for name in valid: register((dest / name).parent)
        data_files.extend(dest.rglob('kpm_tgl_cleaned.csv'))
    if set(bundles) != set(CONDITIONS): raise ValueError('Attach both model bundles or the prepared input package.')
    expected = {json.loads((p/'manifest.json').read_text())['identity']['data_sha256'] for p in bundles.values()}
    if len(expected) != 1: raise ValueError('Models were trained with different dataset identities.')
    matches = [p for p in data_files if digest(p) in expected]
    if not matches: raise ValueError('No matching CSV. Attach the prepared weighted32 input ZIP, including its unchanged matching CSV.')
    print('Matching CSV:', matches[0])
    return bundles, matches[0]

def validate(bundles, data, split):
    if split not in ('test', 'validation'): raise ValueError('Use test or validation, never train for final reporting.')
    infos = {}; training = []
    for condition, path in bundles.items():
        info = json.loads((path/'manifest.json').read_text()); infos[condition] = info
        if info['identity']['condition'] != condition: raise ValueError('Condition mismatch')
        expected = EXPECTED_CHECKPOINTS[condition]
        if digest(path/'best_source.safetensors') != expected['weights'] or digest(path/'source_artifact/tokenizer.json') != expected['tokenizer']:
            raise ValueError(f'{condition}: expected the current 30-epoch checkpoint; older checkpoints are not valid inputs.')
        if info['cfg']['epochs'] != 30:
            raise ValueError('This notebook expects a 30-epoch training budget.')
        if condition == 'morph_bpe':
            if (info.get('weighted_penalty_used') is not True or info.get('tokenizer_variant') != 'weighted_penalty32'
                    or info.get('crossing_penalty') != 32):
                raise ValueError('Morph-BPE must be the trained weighted penalty32 model.')
        if digest(data) != info['identity']['data_sha256']: raise ValueError('Dataset checksum mismatch')
        for rel, key in [('source_artifact/tokenizer.json','tokenizer_sha256'),('source_artifact/tokenizer-manifest.json','artifact_manifest_sha256'),('nllb_source_adapter.py','helper_sha256'),('runtime/kapampangan_morphbpe_runtime/tokenizer.py','runtime_sha256')]:
            if digest(path/rel) != info['identity'][key]: raise ValueError(f'Checksum mismatch: {condition}/{rel}')
        artifact = json.loads((path/'source_artifact/tokenizer-manifest.json').read_text())
        for name, sha in artifact['core_file_sha256'].items():
            if Path(name).name != name or digest(path/'source_artifact'/name) != sha: raise ValueError(f'Artifact mismatch: {name}')
        weights = load_file(str(path/'best_source.safetensors'))
        if set(weights) != {'weight'}: raise ValueError('Unexpected source weight keys')
        weight = weights['weight']
        if tuple(weight.shape) != (info['source_vocab_size'], 1024) or not torch.isfinite(weight).all():
            raise ValueError('Invalid source embedding weights')
        history = json.loads((path/'history.json').read_text())
        if [r.get('epoch') for r in history] != list(range(1,31)):
            raise ValueError('Expected all 30 completed epochs in order.')
        if any(not isinstance(r.get(k),(int,float)) or not torch.isfinite(torch.tensor(r[k])).item()
               for r in history for k in ('train_loss','validation_loss')):
            raise ValueError('Invalid training history losses.')
        best = min(history, key=lambda r: r['validation_loss'])
        if best['epoch'] != 30 or info.get('selected_epoch', best['epoch']) != 30:
            raise ValueError('Expected the installed epoch-30 selection.')
        training.append(dict(condition=condition, completed_epochs=max(r['epoch'] for r in history), best_epoch=best['epoch'],
                             train_loss=best['train_loss'], validation_loss=best['validation_loss'],
                             weighted_penalty_used=info.get('weighted_penalty_used', artifact.get('metadata',{}).get('weighted_penalty_used')),
                             tokenizer_variant=info.get('tokenizer_variant', 'plain_bpe'),
                             crossing_penalty=info.get('crossing_penalty')))
        del weights, weight
    a,b = (infos[c] for c in CONDITIONS)
    for key in ('model_id','revision','source_vocab_size','adaptation','target_language','source_id_mapping'):
        if a[key] != b[key]: raise ValueError(f'Unmatched model setting: {key}')
    for key in ('seed','lr','batch_size','accumulation','initialization','epochs','early_stopping_patience','early_stopping_min_delta'):
        if a['cfg'][key] != b['cfg'][key]: raise ValueError(f'Unmatched training setting: {key}')
    for key in ('source_limit','target_limit','beams','max_new_tokens','data_policy','excluded_train_ids'):
        if a['identity'][key] != b['identity'][key]: raise ValueError(f'Unmatched condition setting: {key}')
    if digest(bundles[CONDITIONS[0]]/'target_tokenizer/tokenizer.json') != digest(bundles[CONDITIONS[1]]/'target_tokenizer/tokenizer.json'):
        raise ValueError('Target vocabularies differ')
    with Path(data).open(encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        if not {'id','source','target','split','group_id'}.issubset(reader.fieldnames or []): raise ValueError('CSV schema mismatch')
        rows = [r for r in reader if r['split'] == split]
    if not rows or len({r['id'] for r in rows}) != len(rows): raise ValueError('Empty split or duplicate IDs')
    if any(not r['source'].strip() or not r['target'].strip() for r in rows): raise ValueError('Empty source/reference')
    for path in bundles.values():
        saved = json.loads((path/'split_manifest.json').read_text())
        if [r['id'] for r in rows] != saved[split]: raise ValueError('Rows differ from saved evaluation split')
    print('Split:', split, '| rows:', len(rows)); print(json.dumps(training, indent=2))
    return infos, rows, training

def restore(input_root, work):
    previous = list(Path(input_root).rglob('paired_evaluation_v2_backup.zip'))
    if len(previous) > 1: raise ValueError('Attach at most one progress backup')
    if previous:
        if list(Path(work).glob('*/*.jsonl')): raise ValueError('Working progress already exists; do not overwrite it')
        extract(previous[0], work)
    print('Resume backup:', previous)

def load_helper(path, condition):
    # Only execute code from your own trusted, checksum-validated bundles.
    spec = importlib.util.spec_from_file_location('saved_adapter_'+condition, path/'nllb_source_adapter.py')
    helper = importlib.util.module_from_spec(spec); spec.loader.exec_module(helper)
    return helper

def read_progress(path, rows):
    completed = []
    if path.exists():
        lines = path.read_text(encoding='utf-8').splitlines()
        for i,line in enumerate(lines):
            try: record = json.loads(line)
            except json.JSONDecodeError:
                if i != len(lines)-1: raise ValueError('Corrupt interior progress record')
                break
            j = len(completed)
            if j >= len(rows) or any(record[k] != rows[j][k] for k in ('id','source','target')) or not isinstance(record.get('prediction'),str):
                raise ValueError('Prediction alignment mismatch')
            completed.append(record)
        path.write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in completed),encoding='utf-8')
    return completed

def corpus_metrics(hypotheses, references):
    if len(hypotheses) != len(references) or not hypotheses: raise ValueError('Unaligned or empty evaluation')
    result = {}
    for name, metric in [('BLEU', BLEU(tokenize='13a', lowercase=False, smooth_method='exp', effective_order=False)),
                         ('chrF++', CHRF(char_order=6, word_order=2, beta=2, lowercase=False))]:
        score = metric.corpus_score(hypotheses, [references])
        result[name] = dict(score=score.score, signature=str(metric.get_signature()))
    return result

def archive(work):
    return shutil.make_archive(str(Path(work).parent/'paired_evaluation_v2_backup'), 'zip', root_dir=work)

def evaluate(bundles, infos, rows, training, work, split, max_examples=None):
    work = Path(work); work.mkdir(parents=True,exist_ok=True)
    all_rows = rows
    if max_examples is not None:
        if not isinstance(max_examples,int) or max_examples < 1: raise ValueError('MAX_EXAMPLES must be None or a positive integer')
        rows = rows[:max_examples]
    final = max_examples is None
    save_json(work/'training_diagnostics.json', training)
    summary = []
    torch.backends.cuda.matmul.allow_tf32 = False
    for condition in CONDITIONS:
        path=bundles[condition]; info=infos[condition]; out=work/condition; out.mkdir(exist_ok=True)
        identity = dict(condition=condition, split=split, final_full_split=final, examples=len(rows),
                        row_ids_sha256=hashlib.sha256(json.dumps([r['id'] for r in rows]).encode()).hexdigest(),
                        data=info['identity']['data_sha256'], weights=digest(path/'best_source.safetensors'),
                        tokenizer=info['identity']['tokenizer_sha256'], helper=digest(path/'nllb_source_adapter.py'),
                        runtime=info['identity']['runtime_sha256'], evaluator=digest(__file__), base=info['revision'],
                        target=digest(path/'target_tokenizer/tokenizer.json'), beams=info['identity']['beams'],
                        max_new_tokens=info['identity']['max_new_tokens'], source_limit=info['identity']['source_limit'],
                        seed=info['cfg']['seed'], precision='fp32', torch=str(torch.__version__),
                        transformers=transformers.__version__, sacrebleu=sacrebleu.__version__,
                        tokenizer_variant=info.get('tokenizer_variant', 'plain_bpe'),
                        weighted_penalty_used=info.get('weighted_penalty_used', False),
                        crossing_penalty=info.get('crossing_penalty'))
        identity_file=out/'evaluation_identity.json'
        if identity_file.exists() and json.loads(identity_file.read_text()) != identity:
            raise ValueError('Resume identity changed. Use a fresh OUTPUT_NAME, or restore the exact previous settings.')
        save_json(identity_file, identity)
        prediction_file=out/'predictions.jsonl'; completed=read_progress(prediction_file, rows)
        model=source=target=None
        try:
            if len(completed)<len(rows):
                helper=load_helper(path, condition)
                model,source,target=helper.load_bundle(path, 'cuda')
                model.requires_grad_(False); model.eval()
                model.config.use_cache=True
                # Use generation_config directly, avoiding the deprecated config override path.
                model.generation_config=transformers.GenerationConfig.from_model_config(model.config)
                torch.manual_seed(info['cfg']['seed'])
                with prediction_file.open('a',encoding='utf-8') as f:
                    for row in tqdm(rows[len(completed):], initial=len(completed),total=len(rows),desc=condition):
                        ids=source.encode(row['source'])
                        if len(ids)>identity['source_limit']: raise ValueError(f"Overlength source {row['id']}; no silent truncation")
                        x=torch.tensor([ids],device='cuda')
                        with torch.inference_mode():
                            output=model.generate(input_ids=x,attention_mask=x.ne(source.pad).long(),
                                forced_bos_token_id=target.convert_tokens_to_ids('tgl_Latn'),
                                max_new_tokens=identity['max_new_tokens'],num_beams=identity['beams'],do_sample=False,use_cache=True)
                        record=dict(id=row['id'],source=row['source'],target=row['target'],
                                    prediction=target.decode(output[0],skip_special_tokens=True),
                                    reached_output_limit=len(output[0])-1 >= identity['max_new_tokens'] and output[0,-1].item()!=target.eos_token_id)
                        f.write(json.dumps(record,ensure_ascii=False)+'\n');f.flush();completed.append(record)
                        del x,output
                        if len(completed)%100==0: archive(work)
        finally:
            del model,source,target
            gc.collect();torch.cuda.empty_cache();archive(work)
        hypotheses=[r['prediction'] for r in completed];references=[r['target'] for r in rows]
        metrics=corpus_metrics(hypotheses,references)
        metrics['scope']=dict(split=split,examples=len(rows),full_split=final,available_rows=len(all_rows),
                              reached_output_limit=sum(r.get('reached_output_limit',False) for r in completed))
        save_json(out/'metrics.json',metrics)
        (out/'hypotheses.txt').write_text('\n'.join(hypotheses)+'\n',encoding='utf-8')
        (out/'references.txt').write_text('\n'.join(references)+'\n',encoding='utf-8')
        summary.append(dict(condition=condition,split=split,examples=len(rows),full_split=final,
                            BLEU=metrics['BLEU']['score'],chrF_plus_plus=metrics['chrF++']['score']))
        print(summary[-1]);archive(work)
    with (work/'comparison.csv').open('w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=list(summary[0]));writer.writeheader();writer.writerows(summary)
    predictions={c:read_progress(work/c/'predictions.jsonl',rows) for c in CONDITIONS}
    with (work/'paired_predictions.csv').open('w',newline='',encoding='utf-8-sig') as f:
        writer=csv.DictWriter(f,fieldnames=['id','source','reference','plain_bpe','morph_bpe']);writer.writeheader()
        for i,row in enumerate(rows): writer.writerow(dict(id=row['id'],source=row['source'],reference=row['target'],
            **{c:predictions[c][i]['prediction'] for c in CONDITIONS}))
    save_json(work/'comparison.json',dict(rows=summary,morph_minus_plain={key:summary[1][key]-summary[0][key] for key in ['BLEU','chrF_plus_plus']},
        note='Corpus scores on a 0-100 scale. Differences alone do not establish statistical significance or human correctness.'))
    print('Backup:',archive(work))
    return summary


## 4. Discover inputs and verify both models
Checks tokenizer/helper/runtime checksums, finite 6080x1024 source weights, matching base/target vocabulary/settings and exact saved split IDs. Prints actual completed and selected epochs from history. It rejects the older CSV instead of evaluating the wrong split.

In [ ]:
import sys, importlib
sys.path.insert(0, '/kaggle/working')
import paired_evaluation_v2 as evaluation
importlib.reload(evaluation)
bundles, DATA = evaluation.discover(INPUT, WORK)
infos, rows, training = evaluation.validate(bundles, DATA, SPLIT)


## 5. Restore saved progress (optional)
For the first weighted32 evaluation, attach no progress backup. This cell will report an empty resume list and continue. For a later resume, attach only paired_evaluation_v2_backup.zip produced by this revised notebook with the same input ZIP. The previous hard-constrained backup has different weights, tokenizer, runtime and evaluator identities and cannot be resumed.


In [ ]:
evaluation.restore(INPUT, WORK)


## 6. Generate both conditions and compute BLEU / chrF++
One sentence at a time, FP32, same saved beam count and output limit. Each prediction is flushed immediately, and the progress archive is refreshed every 100 sentences, on condition completion and on ordinary interruption/error. A hard session termination can prevent the final archive refresh: save/download progress before ending the session.

BLEU: corpus BLEU, case-sensitive, 13a tokenizer, exponential smoothing. chrF++: character order 6 + word order 2, beta 2, case-sensitive. Metric signatures are exported. Both scores use a 0-100 scale. This is not training, and scores do not establish statistical significance or human correctness.

If out of memory: restart to free other loaded models, then resume with the same settings. Do not change beams or precision for only one condition.

In [ ]:
summary = evaluation.evaluate(bundles, infos, rows, training, WORK, SPLIT, MAX_EXAMPLES)
import pandas as pd
display(pd.DataFrame(summary))


## 6B. Thesis tables: BLEU and chrF++ variables
Run after both conditions complete. This reads saved predictions without reloading NLLB or generating again. It exports BP, corpus lengths, all four BLEU precisions and weights, chrF++ beta/P/R, and n-gram counts. P/R/p_n are 0-1; final scores are 0-100. The initial paper BLEU-2/character-unigram examples are illustrative rather than these complete corpus settings.

In [ ]:
%%writefile /kaggle/working/metric_variable_tables.py
"""Report the sufficient statistics behind saved corpus BLEU and chrF++.

SacreBLEU 2.5.1 is pinned because CHRF corpus statistics use its private API.
No model loading, training or translation occurs here.
"""
import csv
import json
import math
from pathlib import Path
import sacrebleu
from sacrebleu.metrics import BLEU, CHRF

def write_csv(path, rows):
    with Path(path).open('w', newline='', encoding='utf-8-sig') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]))
        writer.writeheader();writer.writerows(rows)

def compute_variables(hypotheses,references):
    if sacrebleu.__version__!='2.5.1':raise ValueError('Use sacrebleu==2.5.1 for this statistics exporter.')
    if not hypotheses or len(hypotheses)!=len(references):raise ValueError('Empty or unaligned predictions.')
    bleu=BLEU(tokenize='13a',lowercase=False,smooth_method='exp',effective_order=False)
    b=bleu.corpus_score(hypotheses,[references])
    p=[v/100 for v in b.precisions]
    geometric=math.prod(p)**0.25
    assert math.isclose(100*b.bp*geometric,b.score,abs_tol=1e-8)
    bleu_row=dict(BP=b.bp,c=b.sys_len,r=b.ref_len,N=4,w1=0.25,w2=0.25,w3=0.25,w4=0.25,
                  p1=p[0],p2=p[1],p3=p[2],p4=p[3],geometric_precision=geometric,BLEU=b.score)
    orders=[]
    for i in range(4):
        orders.append(dict(metric='BLEU',unit='word',order=i+1,hypothesis_count=b.totals[i],reference_count='',
                           matching_count=b.counts[i],raw_precision=b.counts[i]/b.totals[i] if b.totals[i] else 0,
                           scoring_precision=p[i],recall='',included=True))
    chrf=CHRF(char_order=6,word_order=2,beta=2,lowercase=False,whitespace=False,eps_smoothing=False)
    score=chrf.corpus_score(hypotheses,[references])
    stats=chrf._extract_corpus_statistics(hypotheses,[references])
    aggregate=[sum(column) for column in zip(*stats)]
    precision=recall=0.0;effective=0
    for i in range(chrf.order):
        h,r,m=aggregate[3*i:3*i+3]
        valid=h>0 and r>0
        order_p=m/h if h else 0.0;order_r=m/r if r else 0.0
        if valid:precision+=order_p;recall+=order_r;effective+=1
        orders.append(dict(metric='chrF++',unit='character' if i<6 else 'word',order=i+1 if i<6 else i-5,
                           hypothesis_count=h,reference_count=r,matching_count=m,raw_precision=order_p,
                           scoring_precision=order_p,recall=order_r,included=valid))
    if effective:precision/=effective;recall/=effective
    reconstructed=100*5*precision*recall/(4*precision+recall) if precision+recall else 0
    assert math.isclose(reconstructed,score.score,abs_tol=1e-8)
    chrf_row=dict(beta=2,char_order=6,word_order=2,effective_orders=effective,P=precision,R=recall,chrF_plus_plus=score.score)
    return bleu_row,chrf_row,orders,dict(BLEU=str(bleu.get_signature()),chrF_plus_plus=str(chrf.get_signature()))

def export_tables(work):
    work=Path(work);bleu_rows=[];chrf_rows=[];order_rows=[];identities=[]
    for condition in ('plain_bpe','morph_bpe'):
        out=work/condition
        identity=json.loads((out/'evaluation_identity.json').read_text(encoding='utf-8'))
        records=[json.loads(line) for line in (out/'predictions.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
        if len(records)!=identity['examples']:raise ValueError(f'{condition} is incomplete; finish evaluation before reporting tables.')
        identities.append(identity)
        b,c,orders,signatures=compute_variables([r['prediction'] for r in records],[r['target'] for r in records])
        bleu_rows.append(dict(condition=condition,split=identity['split'],examples=len(records),full_split=identity['final_full_split'],**b))
        chrf_rows.append(dict(condition=condition,split=identity['split'],examples=len(records),full_split=identity['final_full_split'],**c))
        order_rows.extend(dict(condition=condition,**r) for r in orders)
        saved=json.loads((out/'metrics.json').read_text(encoding='utf-8'))
        assert math.isclose(b['BLEU'],saved['BLEU']['score'],abs_tol=1e-8)
        assert math.isclose(c['chrF_plus_plus'],saved['chrF++']['score'],abs_tol=1e-8)
        (out/'metric_variables.json').write_text(json.dumps(dict(BLEU=b,chrF_plus_plus=c,ngrams=orders,signatures=signatures),indent=2),encoding='utf-8')
    for key in ('row_ids_sha256','split','examples','data','final_full_split'):
        if identities[0][key]!=identities[1][key]:raise ValueError('Models have unmatched reporting rows.')
    write_csv(work/'bleu_variables.csv',bleu_rows)
    write_csv(work/'chrf_variables.csv',chrf_rows)
    write_csv(work/'ngram_counts.csv',order_rows)
    lines=['# Translation metric variable results','',
        'P, R and p_n use a 0-1 scale; BLEU and chrF++ use a 0-100 scale. BLEU p_n includes the configured exponential smoothing. Corpus lengths/counts are aggregated within sentences, without joining sentence boundaries.','',
        '| Condition | BP | w_n (each) | p1 | p2 | p3 | p4 | BLEU |',
        '| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |']
    for row in bleu_rows:
        lines.append('| '+row['condition']+' | '+' | '.join(f'{row[k]:.6f}' for k in ('BP','w1','p1','p2','p3','p4','BLEU'))+' |')
    lines+=['','| Condition | beta | P | R | chrF++ |','| --- | ---: | ---: | ---: | ---: |']
    for row in chrf_rows:
        lines.append('| '+row['condition']+' | '+' | '.join(f'{row[k]:.6f}' for k in ('beta','P','R','chrF_plus_plus'))+' |')
    lines+=['',f"Split: {identities[0]['split']}; examples: {identities[0]['examples']}; full split: {identities[0]['final_full_split']}.",
        'Results are reference agreement, not a statistical significance or human correctness judgment.']
    (work/'metric_variable_tables.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
    return bleu_rows,chrf_rows


In [ ]:
import pandas as pd
import metric_variable_tables as tables
importlib.reload(tables)
bleu_table, chrf_table = tables.export_tables(WORK)
display(pd.DataFrame(bleu_table))
display(pd.DataFrame(chrf_table))
evaluation.archive(WORK)


## 7. Download results or partial progress
Run after completion, or after interrupting section 6 to save progress. Download the ZIP through the Output file browser or the link below. Save Version can preserve notebook outputs; saving a draft alone is not a backup of working files.

Outputs: comparison.csv and comparison.json; paired_predictions.csv with source/reference/both translations; each model's metrics.json with signatures, hypotheses.txt, references.txt, predictions.jsonl and evaluation_identity.json; training_diagnostics.json. Partial progress has no final paired score until both conditions finish.

In [ ]:
from IPython.display import FileLink, display
backup = evaluation.archive(WORK)
print('Download:', backup)
display(FileLink(backup))


## Interpreting results
Compare corpus scores for the same full split. Higher scores mean greater agreement with these references under these metrics, not guaranteed grammatical correctness. Inspect paired_predictions.csv and have bilingual reviewers assess errors. If both conditions are close, report the measured difference without claiming superiority or equivalence; paired significance testing and repeated training seeds are separate work.

The checked current bundles both completed/selected epoch 30; Morph-BPE is weighted penalty32. Tokenizer-training-corpus overlap with the translation test set remains unaudited. Reference quality is not established by checksums. This revised notebook has been checked locally without fresh NLLB generation. Kaggle execution and measured weighted32 scores remain to be obtained.

Sources: [SacreBLEU metric definitions](https://github.com/mjpost/sacrebleu), [Kaggle model inputs](https://www.kaggle.com/docs/models), [Kaggle datasets](https://www.kaggle.com/docs/datasets).